# General Description

The following notebook contains the code to create, train, validate, and test a rainfall-runoff model using an LSTM network architecture. To run the experiments we will use a subset of the CAMELS-DE dataset.

In [ ]:
import os

COLAB_ROOT = "/content"
REPO_NAME = "DeepLearningHydrology-Course"
REPO_PATH = os.path.join(COLAB_ROOT, REPO_NAME)
os.chdir(COLAB_ROOT)

if not os.path.exists(REPO_PATH):
    print(f"Cloning {REPO_NAME} to get access to data and examples...")
    !git clone https://github.com/eduardoAcunaEspinoza/DeepLearningHydrology-Course.git

os.chdir(REPO_PATH)
print("Current working directory:", os.getcwd())

# Install the Hy2DL library using uv
print("Installing Hy2DL and dependencies with uv...")
!pip install uv --quiet
!uv pip install hy2dl --system

# 4. Download subset of CAMELS-DE from Zenodo
if not os.path.exists("./data/CAMELS-DE"):
    print("Downloading CAMELS-DE subset from Zenodo...")
    !wget -O camels_de.zip "https://zenodo.org/records/22085206/files/CAMELS-DE-subset.zip?download=1" --quiet
    !unzip -q -o camels_de.zip -d ./data/
    !rm camels_de.zip # Clean up the zip file to save space

print("Setup complete and dataset is ready!")

In [2]:
import datetime
import random
import shutil
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
import xarray as xr

from hy2dl.datasetzoo import get_dataset
from hy2dl.evaluation import calculate_metrics, get_tester
from hy2dl.modelzoo import get_model
from hy2dl.training.basetrainer import BaseTrainer
from hy2dl.utils.config import Config

base_dir = Path.cwd().resolve()
color_palette = {"observed": "#377eb8", "simulated": "#4daf4a"}

Part 1. Initialize information

In [17]:
# Path to .yml file where the experiment settings are stored.
path_experiment_settings = "examples/configs/camels_de.yml"

# Read experiment settings
config = Config(path_experiment_settings, base_dir=base_dir)
config.init_experiment()
config.dump()

Dataset = get_dataset(config)
Tester = get_tester(config)

Part 2. Create datasets and dataloaders used to train/validate the model

In [18]:
# Create training dataset
training_dataset = Dataset(cfg=config, time_period="training")
training_dataset.setup_dataset()
# Initialize training object
trainer = BaseTrainer(cfg=config, training_dataset=training_dataset)

2026-08-24 17:27:26 - Creating training dataset in memory...


INFO:distributed.http.proxy:To route to workers diagnostics web server please install jupyter-server-proxy: python -m pip install jupyter-server-proxy
INFO:distributed.scheduler:State start
INFO:distributed.scheduler:  Scheduler at:     tcp://127.0.0.1:40109
INFO:distributed.scheduler:  dashboard at:  http://127.0.0.1:39231/status
INFO:distributed.scheduler:Registering Worker plugin shuffle
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:34315'
INFO:distributed.nanny:        Start Nanny at: 'tcp://127.0.0.1:42653'
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:42609 name: 0
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:42609
INFO:distributed.core:Starting established connection to tcp://127.0.0.1:59374
INFO:distributed.scheduler:Register worker addr: tcp://127.0.0.1:42675 name: 1
INFO:distributed.scheduler:Starting worker compute stream, tcp://127.0.0.1:42675
INFO:distributed.core:Starting established connection to tcp://12

2026-08-24 17:27:40 - Dataset created successfully.


KeyError: "['area', 'elev_mean'] not in index"

trainer is an instance of a class that helps us handle everything that our model

In [ ]:
validation_dataset = Dataset(cfg=config, time_period="validation")
validation_dataset.setup_dataset(check_nan=False, path_scaler = config.path_save_folder / "scaler.yml" )
tester_validation = Tester(cfg=config, evaluation_dataset=validation_dataset)

Part 3. Train model

In [ ]:
# Training report structure
validation_headers = "".join([f"{m:^10}|" for m in config.validation_metric])
config.logger.info("Training model".center(60, "-"))
config.logger.info(f"{'':^16}|{'Training':^21}|{'Validation':^{(11 * len(config.validation_metric)) + 10}}|")
config.logger.info(f"{'Epoch':^5}|{'LR':^10}|{'Loss':^10}|{'Time':^10}|{validation_headers}{'Time':^10}|")

# Loop through epochs
total_time = time.time()
for epoch in range(1, config.epochs + 1):
    trainer.train_model(epoch=epoch)  # Training
    tester_validation.validate_model(model=trainer.model, epoch=epoch)  # Validation
    config.logger.info(trainer.report + tester_validation.validation_report)  # report

config.logger.info(f"Total training time: {datetime.timedelta(seconds=int(time.time() - total_time))}\n")
shutil.rmtree(tester_validation.path_zarr, ignore_errors=True)  # delete validation results

Part 4. Test model

In [ ]:
# If I already trained a model, I can re-construct it using the saved parameters from a given epoch
model = get_model(config).to(config.device)
model.load_state_dict(torch.load(config.path_save_folder / "model" / f"model_epoch_{config.epochs}", map_location=config.device))

In [ ]:
testing_dataset = Dataset(cfg=config, time_period="testing")
testing_dataset.setup_dataset(check_nan=False, path_scaler = config.path_save_folder / "scaler.yml" )
tester_testing = Tester(cfg=config, evaluation_dataset=testing_dataset)

config.logger.info("Testing model...")
testing_time = time.time()
tester_testing.evaluate_model(model = trainer.model)
config.logger.info("Testing completed.")
config.logger.info(f"Total testing time: {datetime.timedelta(seconds=int(time.time() - testing_time))}\n")

Part 5. Initial analysis

In [ ]:
test_results = xr.open_zarr(tester_testing.path_zarr)
testing_metrics = calculate_metrics(ds_results=test_results, metric_name = config.testing_metrics)
testing_metrics.to_zarr(config.path_save_folder / "testing_metrics.zarr", mode="w")

In [ ]:
# Loss testing
target_of_interest = random.sample(list(testing_metrics.feature.values), 1)[0]
test_metric = testing_metrics.sel(feature=target_of_interest, metric="nse").round(3).T.to_pandas().dropna()
# Plot the histogram
plt.figure(figsize=(10, 5))
plt.hist(test_metric, bins = np.linspace(0.0, 1.0, 11).tolist())
# Add NSE statistics to the plot
plt.text(
    0.01,
    0.8,
    (
        f"Mean: {'%.2f' % test_metric.mean():>7}\n"
        f"Median: {'%.2f' % test_metric.median():>0}\n"
        f"Max: {'%.2f' % test_metric.max():>9}\n"
        f"Min: {'%.2f' % test_metric.min():>10}"
    ),
    transform=plt.gca().transAxes,
    bbox=dict(facecolor="white", alpha=0.5),
)

# Format plot
plt.xlabel("NSE", fontsize=12, fontweight="bold")
plt.ylabel("Frequency", fontsize=12, fontweight="bold")
plt.title(f"NSE histogram for: {target_of_interest}", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# Plot simulated and observed discharges
basin_to_analyze = random.sample(list(test_results.gauge_id.values), 1)[0]
y_sim = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_sim"].compute().values
y_obs = test_results.sel(gauge_id=basin_to_analyze, feature=target_of_interest)["y_obs"].compute().values

plt.figure(figsize=(15, 7.5))
plt.plot(y_obs, label="observed", color=color_palette["observed"])
plt.plot(y_sim, label="simulated", alpha=0.5, color=color_palette["simulated"])

# Format plot
plt.xlabel("Date", fontsize=12, fontweight="bold")
plt.ylabel(target_of_interest, fontsize=12, fontweight="bold")
plt.title(f"Results for gauge_id: {basin_to_analyze}", fontsize=16, fontweight="bold")
plt.tick_params(axis="both", which="major", labelsize=12)
plt.legend(loc="upper right", fontsize=12)
plt.tight_layout()
plt.show()